# Group A — Can a cheap surrogate replace an expensive sensitivity study?

**AI Capstone project — finding important turbulence-model parameters with explainable AI**

---

## The engineering problem

Computational Fluid Dynamics (CFD) predicts turbulent flows using a *turbulence
model*. The model used here, **SST $k$–$\omega$**, contains nine empirical
constants. Nobody knows their exact values; each is only known to lie inside an
interval. Different choices inside those intervals give different predicted
flows, so an engineer who wants a trustworthy prediction needs to know **which of
the nine constants actually matter**.

The standard way to answer this is a *Sobol sensitivity analysis*: sample the
parameter space many times, run a CFD simulation for every sample, and decompose
the variance of the output. For this dataset that took **1408 CFD simulations per
flow case** — weeks of supercomputer time.

## Your question

> Could we have reached the same conclusion with far fewer simulations, by
> training a machine-learning model on a small subset and asking *it* which
> parameters matter?

This is what Group A investigates. You will:

1. train surrogate models that predict a CFD output from the nine parameters;
2. measure how their accuracy and their recovered parameter ranking degrade as
   you take away training data;
3. compute sensitivity indices *from the surrogate* and compare them against the
   indices computed from the full CFD study.

The Sobol indices from the full study are provided to you as a **benchmark**. You
never have to compute them from CFD — that part is already done.

---

---

## How to use this notebook

This notebook is the **skeleton of your project**, not a worked example. The
text, the data loading and the figures that show you the provided data are all
given. The analysis is yours to write.

Look for blocks like this:

```
# ==========================================================================
# TODO - fit the surrogate model
#
# ... what to do, and why ...
#
# Hints:
#   * ...
#
# Later cells expect:
#   forest : a fitted regressor
# ==========================================================================
```

Each one states the task, gives hints, and — where it matters — names the
variable that later cells will use. Keep those names, or change them
consistently throughout.

Every library function you need is listed, with a link to its documentation and
a worked example, in **[`RESOURCES.md`](../RESOURCES.md)** at the top of the
repository. You are not expected to implement any method yourself — the whole
project is about applying existing implementations and interpreting what they
tell you.

Sections are labelled **MUST**, **SHOULD** or **COULD**:

* **MUST** — the project. Every group does these.
* **SHOULD** — where your group goes deeper. Agree with your supervisor which
  of these you commit to in your project plan; you are not expected to do all.
* **COULD** — optional, and deliberately unscaffolded. If you take one on, you
  design it yourselves.

You are five or six people. Split the work by section, not by everyone trying
the same thing. And if you disagree with a choice made here, change it and say
why when you present — that is a better outcome than following it.

In [ ]:
# --- Setup --------------------------------------------------------------------
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone


def find_project_root(marker="src/capstone_tools.py"):
    """Locate the top of the cloned repository.

    Walks up from the current working directory, so the notebook works whether
    Jupyter was started in notebooks/, in the project root, or anywhere else
    inside the clone. Nothing here depends on where you put the folder.
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / marker).exists():
            return base
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from inside the "
        "cloned repository, so that a 'src/capstone_tools.py' exists in one of "
        "the parent directories of " + str(Path.cwd())
    )


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
import capstone_tools as ct

warnings.filterwarnings("ignore", category=FutureWarning)
ct.set_plot_style()

RANDOM_STATE = 0          # fixed so that your results are reproducible
FIGDIR = ROOT / "figures"
FIGDIR.mkdir(exist_ok=True)

print("project root  :", ROOT)
print("data directory:", ct.DATA_DIR)
print("parameters:", ", ".join(ct.PARAMETER_NAMES))

## 1. The nine turbulence-model parameters

Each parameter was varied inside the interval shown below. These intervals come
from the turbulence-modelling literature: they are the range of values different
authors consider defensible.

In [ ]:
info = ct.load_parameter_info()
info

## 2. The data

We start with **turbulent channel flow at $Re_\tau = 640$** — flow between two
parallel walls, the most studied configuration in all of turbulence research.

For each of the 1408 simulations the CFD solver produced profiles of several
quantities across the channel, sampled at 40 positions. Two of those quantities
matter here:

| symbol | meaning | why it matters |
|---|---|---|
| $p'_{\mathrm{rms}}$ | root-mean-square pressure fluctuation | drives vibration and fatigue of structures in the flow |
| $k$ | turbulent kinetic energy | the central quantity of the turbulence model itself |

From these 40-point profiles we have derived six **scalar outputs** — single
numbers per simulation. These are what you will predict, because a single number
is far easier to model and to explain than a whole profile.

In [ ]:
CASE, RE_TAU = "channel", 640

data = ct.load_scalar_outputs(CASE, RE_TAU)
coords = ct.load_coordinates(CASE, RE_TAU)

print(f"{data.shape[0]} simulations x {len(ct.PARAMETER_NAMES)} parameters")
print("scalar outputs:", ", ".join(ct.SCALAR_OUTPUTS))
data.head()

### What the simulations actually look like

Before modelling anything, look at the data. The grey lines below are individual
CFD simulations: same flow, same solver, only the nine turbulence-model constants
differ. The spread between them *is* the uncertainty we are trying to attribute
to individual parameters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, var in zip(axes, ["p_rms", "k"]):
    prof = ct.load_profiles(CASE, var, RE_TAU)
    values = prof[[c for c in prof.columns if c.startswith("point_")]].to_numpy()
    y = coords["y_plus"].to_numpy()
    for row in values[::20]:
        ax.plot(row, y, color="0.7", lw=0.6, alpha=0.6)
    ax.plot(values.mean(axis=0), y, color="crimson", lw=2.4, label="ensemble mean")
    ax.set_xlabel(ct.VARIABLE_LABELS[var])
    ax.set_ylabel(r"$y^+$ (distance from the wall)")
    ax.set_yscale("log")
    ax.set_title(f"{ct.VARIABLE_LABELS[var]} — {len(values)} simulations")
    ax.legend()
fig.suptitle(f"Spread caused by turbulence-model parameters ({ct.case_label(CASE, RE_TAU)})")
fig.tight_layout()
fig.savefig(FIGDIR / "A01_data_spread.png")
plt.show()

## 3. The benchmark: what the full CFD study found

The Sobol index $S_i$ of parameter $i$ is the fraction of the output variance
that is explained by varying that parameter alone. The indices sum to
(approximately) one, so you can read them as percentages of the variance.

Two kinds are provided:

* **first-order $S_1$** — the effect of a parameter *on its own*;
* **total-order $S_T$** — its effect *including* everything it does in
  combination with other parameters.

This is the answer your workflow is trying to reproduce from much less data.

In [ ]:
bench = ct.load_sobol_scalar(CASE, RE_TAU)
QOI = "p_rms_peak"          # the output we model first

# Permutation importance and SHAP both include whatever a parameter does in
# combination with the others, so the TOTAL-order index is the like-for-like
# comparison. We use ST as the primary reference throughout and keep S1 for the
# interaction discussion in section 12.
BENCHMARK = "ST"

bench_q = (bench[bench.output == QOI]
           .set_index("parameter")
           .reindex(ct.PARAMETER_NAMES))

fig, ax = plt.subplots(figsize=(9, 4))
order = bench_q[BENCHMARK].sort_values(ascending=False).index
idx = np.arange(len(order))
ax.bar(idx - 0.2, bench_q.loc[order, "ST"], width=0.4,
       yerr=bench_q.loc[order, "ST_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], label="$S_T$ (total order)")
ax.bar(idx + 0.2, bench_q.loc[order, "S1"], width=0.4,
       yerr=bench_q.loc[order, "S1_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], alpha=0.45, label="$S_1$ (first order)")
ax.set_xticks(idx)
ax.set_xticklabels([ct.PRETTY[p] for p in order])
ax.set_ylabel("Sobol index")
ax.set_title(f"Benchmark from all 1408 CFD simulations — {QOI}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "A02_benchmark.png")
plt.show()

print(f"sum of S1 = {bench_q.S1.sum():.3f}   sum of ST = {bench_q.ST.sum():.3f}")
print("A sum close to 1 means the parameters act almost independently, so S1 and")
print("ST nearly coincide here - but ST is the one we compare against.")
print("\nbenchmark ranking (by ST):")
for rank, p in enumerate(order, 1):
    print(f"  {rank}. {p:12s} ST = {bench_q.loc[p, 'ST']:.4f}   S1 = {bench_q.loc[p, 'S1']:.4f}")

Two parameters carry almost all of the variance: $\beta^*$ (the $k$-dissipation
coefficient) and $\alpha_{\omega 1}$. Then there is a large gap, and the
remaining seven are near zero.

That gap is the key to the whole project. Identifying the **top two** will turn
out to be easy. Correctly ordering the parameters *in the gap* — $\kappa$,
$\beta_1$, $\alpha_{k1}$, whose indices are a few percent — is the hard part, and
it is where the amount of training data really matters.

---
## 4. MUST — a surrogate model and an honest error estimate

### 4a. Splitting the data: why a random split would mislead you

The 1408 simulations are **not** 1408 independent random samples. They come from
a *Saltelli design*, built in 128 blocks of 11. Within a block, the samples
differ from one another in **only one of the nine parameters** — that structure is
exactly what makes the Sobol estimator work.

The consequence for us: if you split rows at random, almost every test
simulation has a near-twin in the training set, differing in one coordinate out
of nine. Any model will predict those nearly perfectly, and your test score will
flatter you.

The dataset therefore carries a `design_group` column. Keep a whole block on one
side of the split. Let us measure how much difference this makes.

In [ ]:
# ==========================================================================
# TODO - compare a random split with a grouped split
#
# Show for yourselves how much a naive random split flatters the model.
#
# Fit the same Random Forest twice on the same data: once with an ordinary
# random split, once keeping each design_group together. Print the test R2
# from each. Use test_size=0.2 and random_state=RANDOM_STATE for both so the
# comparison is fair.
#
# Write down the two numbers - you will show them when you present.
#
# Hints:
#   * sklearn.model_selection.ShuffleSplit for the random split
#   * sklearn.model_selection.GroupShuffleSplit for the grouped split, passing groups=data['design_group']
#   * next(splitter.split(X, y, groups)) gives (train_idx, test_idx)
#   * X = data[ct.PARAMETER_NAMES].to_numpy(), y = data[QOI].to_numpy()
#
# Later cells expect:
#   X, y, groups : the feature matrix, the target, and the design-group labels
# ==========================================================================


# your code here

### 4b. The surrogate model

We use a Random Forest: a collection of regression trees, each fitted to a
different bootstrap sample of the data, averaged together. It needs almost no
tuning, handles non-linear responses, and is the standard first choice for a
tabular problem of this size.

In [ ]:
# ==========================================================================
# TODO - fit the surrogate and measure how good it is
#
# Make the grouped split you will use for the rest of the notebook, fit a
# Random Forest, and report the three standard regression metrics on the test
# set: RMSE, normalised RMSE (RMSE divided by the standard deviation of the
# test target) and R2.
#
# Normalised RMSE is the one to quote in a talk: it says what fraction of the
# output's natural spread your error represents, so it needs no units.
#
# Hints:
#   * RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
#   * sklearn.metrics.mean_squared_error and r2_score
#   * a small evaluate() helper will save you repeating this later
#
# Later cells expect:
#   X_train, X_test, y_train, y_test : the grouped split
#   train_idx, test_idx              : the corresponding index arrays
#   forest                           : the fitted Random Forest
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - draw the parity plot
#
# Scatter the CFD values against your surrogate's predictions on the test set,
# with a diagonal line for reference. Put the R2 in the title.
#
# This is the single figure that persuades an audience your surrogate works,
# so make it readable: square axes, equal ranges, a visible diagonal.
#
# Hints:
#   * ax.plot(lims, lims, 'k--') for the diagonal
#   * save it with fig.savefig(FIGDIR / 'A03_parity.png')
# ==========================================================================


# your code here

---
## 5. MUST — explaining the surrogate with permutation feature importance

The surrogate is accurate, but it is a black box: 300 trees tell us nothing
directly about which parameter matters.

**Permutation feature importance** answers that by breaking one parameter at a
time. Shuffle the values of a single column in the *test* set, so the parameter
keeps its distribution but loses its relationship to the output, and re-score the
model. If the score collapses, the model relied on that parameter; if nothing
happens, it did not.

We shuffle several times (`n_repeats`) because each shuffle is random, which also
gives us a spread to quote as an error bar.

In [ ]:
# ==========================================================================
# TODO - rank the parameters with permutation feature importance
#
# Apply permutation feature importance to the fitted surrogate, on the TEST
# set, scoring with R2. Use enough repeats (30 is plenty) that you also get a
# spread to quote as an error bar.
#
# Collect the result into a tidy table sorted from most to least important.
#
# Hints:
#   * sklearn.inspection.permutation_importance
#   * it returns .importances_mean and .importances_std
#   * n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1, scoring='r2'
#   * why the test set and not the training set? section 9 of the Group B notebook answers that - think about it before you read it
#
# Later cells expect:
#   importance : DataFrame with columns ['parameter', 'importance', 'spread'],
#                sorted by importance descending
# ==========================================================================


# your code here

### Comparing the explanation with the benchmark

Permutation importance is measured in *drop in $R^2$*, while a Sobol index is a
*fraction of variance*. The two are not on the same scale, so comparing their raw
values is meaningless — what we compare is the **ranking**, and the shape of the
profile across parameters.

In [ ]:
# ==========================================================================
# TODO - compare your ranking with the benchmark
#
# Put your permutation importances next to the benchmark Sobol indices.
#
# A warning before you plot: permutation importance is measured as a DROP IN
# R2, while a Sobol index is a FRACTION OF VARIANCE. They are not on the same
# scale and a shared axis would be meaningless. Draw two panels, each with its
# own y-axis, and compare the ORDER and the SHAPE rather than the values.
#
# Print both rankings underneath so you can read them off.
#
# Hints:
#   * join importance (indexed by parameter) with bench_q[['S1','ST']]
#   * reindex to ct.PARAMETER_NAMES so the order is always the same
#   * colour the bars with ct.COLOURS[p] so a parameter keeps its colour in every figure you make
#
# Later cells expect:
#   merged : DataFrame indexed by parameter, with at least the columns
#            'importance', 'S1', 'ST', and 'benchmark' (= merged[BENCHMARK])
# ==========================================================================


# your code here

---
## 6. SHOULD — put a number on the agreement

Comparing two bar charts by eye is not a result. Three standard measures:

* **Spearman rank correlation** over all nine parameters — but beware: seven of
  the nine have benchmark indices near zero, so their *relative* order is
  essentially noise, and this number is dominated by that noise. Expect it to
  look mediocre and to stop improving with more data.
* **Top-$k$ overlap** — how many of the benchmark's $k$ most important
  parameters appear in the AI method's top $k$. This is the measure an engineer
  cares about, because the point of the exercise is to decide which parameters
  to keep.
* **Kendall's $\tau$** — like Spearman, but based on counting pairwise
  disagreements.

In [ ]:
# ==========================================================================
# TODO - turn the comparison into a number
#
# Write a small function that compares an importance vector against a
# benchmark vector and returns a dictionary of agreement measures:
#
#   * Spearman rank correlation over all nine parameters
#   * Kendall's tau
#   * top-k overlap for k = 2, 3, 4, i.e. what fraction of the benchmark's top
#     k parameters appear in your top k
#
# Apply it to your result and print the numbers, then print the two top-4 sets
# side by side.
#
# Expect the Spearman number to look disappointing. Before you conclude your
# method failed, look at the benchmark values for the bottom five parameters
# and ask what their 'true' order even means.
#
# Hints:
#   * scipy.stats.spearmanr(...).correlation and kendalltau(...).correlation
#   * np.argsort(-vector) gives indices from largest to smallest
#   * set(a[:k]) & set(b[:k]) gives the overlap
#
# Later cells expect:
#   rank_agreement(scores, reference, ks=(2, 3, 4)) -> dict
#       used in several later sections, so keep this signature
# ==========================================================================


# your code here

## 7. SHOULD — are the parameters really separated?

An importance score without an uncertainty is not interpretable. Two sources of
spread matter, and they are different things:

1. the **shuffling** spread, which `permutation_importance` already returns;
2. the spread caused by **which simulations landed in the training set**, which
   you get by repeating the whole fit on several different grouped splits.

The second is usually much larger — and it is the one that tells you whether a
conclusion would survive if you had run a different set of simulations.

In [ ]:
# ==========================================================================
# TODO - put error bars on the importance scores
#
# A single importance number is not interpretable on its own. Repeat the whole
# fit-and-explain procedure over several different grouped splits (8 is
# enough) and report the mean and standard deviation across those repeats.
#
# Then plot it as a bar chart with error bars, and state which parameters are
# separated by more than their error bars - those are the ones you can defend.
#
# Note that this spread is a different and usually larger thing than the
# repeat-to-repeat spread that permutation_importance returns internally. Make
# sure you understand which one you are showing.
#
# Hints:
#   * GroupShuffleSplit(n_splits=8, test_size=0.2, random_state=RANDOM_STATE)
#   * iterate over splitter.split(X, y, groups)
#   * collect each repeat as a pd.Series indexed by parameter, then pd.DataFrame(rows) and .mean() / .std()
#
# Later cells expect:
#   summary : DataFrame indexed by parameter with columns ['mean', 'std'],
#             sorted by mean descending
# ==========================================================================


# your code here

---
## 8. SHOULD — how little CFD data is enough? (the core Group A result)

Now the central experiment. Train on a *subset* of the simulations and see what
survives. Two rules matter:

* subsample **whole design groups**, never individual rows, for the reason given
  in section 4a;
* repeat each size several times with different random subsets, because a single
  small subset can be lucky or unlucky.

We track two things against the number of simulations: how well the surrogate
predicts, and how well its explanation matches the benchmark.

In [ ]:
# ==========================================================================
# TODO - the core experiment: how little CFD data is enough?
#
# This is the central result of the Group A project. Train on progressively
# smaller subsets and record both how well the surrogate predicts and how well
# its explanation still matches the benchmark.
#
# Two rules that matter:
#
#   1. Subsample whole DESIGN GROUPS, never individual rows. A group is 11
#      simulations, so n_groups=36 means 396 simulations.
#   2. Repeat each size several times with different random subsets, because a
#      single small subset can be lucky. Five repeats is enough.
#
# For each (size, repeat): select the training groups, fit, score R2 on
# everything not selected, compute permutation importance, and measure the
# agreement with the benchmark using your function from section 6.
#
# Hints:
#   * train = np.isin(groups, chosen_group_ids) gives a boolean mask
#   * use ~train for the evaluation set
#   * sklearn.base.clone(model) to get a fresh unfitted copy each time
#   * suggested sizes: [4, 9, 18, 36, 73, 110] groups
#   * drop n_repeats to about 10 inside permutation_importance here, otherwise this cell takes a long time
#
# Later cells expect:
#   curve      : tidy DataFrame, one row per (size, repeat), with at least
#                'n_simulations', 'R2', 'spearman', 'top2_overlap', 'top4_overlap'
#   curve_mean : curve grouped by n_simulations, mean
#   curve_std  : the same, standard deviation
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - plot the learning curves
#
# Two panels side by side:
#
#   left  - surrogate accuracy (R2) against the number of training simulations
#   right - agreement with the benchmark: top-2 overlap, top-4 overlap and
#           Spearman, all on one axis from 0 to 1
#
# Use a logarithmic x-axis and error bars from curve_std.
#
# Then look at it properly. The three curves on the right do NOT behave the
# same way, and explaining why is worth more marks than the figure itself.
#
# Hints:
#   * ax.errorbar(x, y, yerr=..., marker='o', capsize=3)
#   * ax.set_xscale('log')
# ==========================================================================


# your code here

Read this figure carefully — it contains the two most important messages of the
project, and they point in different directions.

* **Accuracy** keeps improving with more data, as you would expect.
* **Top-2 overlap** is already perfect with a few dozen simulations. The two
  dominant parameters are so dominant that almost no data is needed to find them.
* **Spearman over all nine** stays mediocre and does *not* improve. That is not a
  failure of the method: the seven unimportant parameters have nothing to rank, so
  their order is noise, and that noise never goes away.

A good presentation says all three, and does not quote the Spearman number on
its own as if it measured success.

## 9. SHOULD — is a different surrogate better?

Random Forest was an arbitrary first choice. Compare it with gradient boosting,
which builds trees sequentially, each correcting the previous one's errors.
Both are trained on exactly the same data so the comparison is fair.

In [ ]:
# ==========================================================================
# TODO - is a different surrogate better?
#
# Random Forest was an arbitrary first choice. Compare it against gradient
# boosting on exactly the same training data, and against XGBoost if you have
# it installed.
#
# For each model report the test R2, the normalised RMSE, the top-4 overlap
# with the benchmark, and the first four parameters of its ranking. Put it in
# one table.
#
# State a conclusion: which model would you recommend to an engineer who can
# afford only a few hundred CFD simulations?
#
# Hints:
#   * sklearn.ensemble.GradientBoostingRegressor
#   * wrap the xgboost import in try/except so the notebook still runs without it
#   * clone() each model before fitting so the comparison is clean
# ==========================================================================


# your code here

## 10. SHOULD — sensitivity indices from the surrogate instead of from CFD

This is the sharpest experiment available in this dataset, and it is only a few
lines of code.

A Sobol analysis needs thousands of model evaluations. CFD cannot afford that —
but a *surrogate* can be evaluated millions of times for free. So:

1. train the surrogate on a few hundred CFD simulations;
2. draw a fresh Saltelli sample of the parameter space;
3. evaluate the **surrogate** on it (not CFD);
4. run the ordinary Sobol estimator on those surrogate predictions.

The result is a set of Sobol indices obtained for the price of a few hundred
simulations, which we can hold directly against the indices from all 1408.

In [ ]:
# ==========================================================================
# TODO - compute Sobol indices from the surrogate instead of from CFD
#
# This is the sharpest experiment in the project and it is only a few lines.
#
# A Sobol analysis needs thousands of model evaluations, which CFD cannot
# afford - but a surrogate can be evaluated millions of times for nothing. So:
#
#   1. train the surrogate on a few hundred CFD simulations;
#   2. draw a fresh Saltelli sample of the nine-parameter space;
#   3. evaluate the SURROGATE on that sample (not CFD);
#   4. run the ordinary Sobol estimator on those predictions.
#
# Do this for three training budgets (try 18, 36 and 73 design groups) and
# collect the results next to the CFD benchmark and your permutation
# importances in one table.
#
# Hints:
#   * from SALib.sample import sobol as sobol_sample
#   * from SALib.analyze import sobol as sobol_analyze
#   * problem = ct.make_salib_problem()
#   * sobol_sample.sample(problem, 2048, calc_second_order=False)
#   * sobol_analyze.analyze(..., calc_second_order=False) returns both 'S1' and 'ST' - use the same index as BENCHMARK
#
# Later cells expect:
#   surrogate_sobol(model, problem) -> pd.Series indexed by parameter
#   three_way : DataFrame with the CFD benchmark, your permutation importances,
#               and one column per training budget
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - plot the three-way comparison
#
# A grouped bar chart: parameters on the x-axis, one bar group per source (CFD
# benchmark, then each surrogate budget).
#
# Underneath, print the top-4 parameter set from each source and mark which
# ones match the benchmark exactly. That table is the result; the chart just
# makes it visible.
#
# Hints:
#   * offset each series by j*width along np.arange(len(order))
#   * sort the parameters by the benchmark so the chart reads left to right
# ==========================================================================


# your code here

## 11. SHOULD — the smallest usable number of simulations, and what it saves

The original CFD study reduced the nine parameters to a four-parameter subset
before doing anything expensive with them. So the operational question is: **how many CFD
simulations do you need before the AI workflow identifies that same subset?**

In [ ]:
# ==========================================================================
# TODO - find the smallest usable number of simulations
#
# The published study reduced the nine parameters to a four-parameter subset
# before doing anything expensive. So the operational question is: how many
# CFD simulations does the AI workflow need before it identifies that same
# subset?
#
# For each training-set size, repeat several times and count how often the
# surrogate-based top-4 set exactly equals the benchmark top-4 set. Report
# that success rate against the number of simulations.
#
# Then state the result in the form an engineer wants: "N simulations instead
# of 1408", and what factor that saves.
#
# Hints:
#   * target = set of the benchmark's top four parameter names
#   * compare with == on sets, not on lists - order does not matter here
#   * reuse surrogate_sobol() from the previous section
# ==========================================================================


# your code here

## 12. SHOULD — do the parameters interact?

Compare the first-order index $S_1$ (a parameter acting alone) with the
total-order index $S_T$ (the same parameter including every interaction it takes
part in). The difference $S_T - S_1$ is the part of the variance that parameter
only produces *in combination* with others.

This matters for interpreting your own results: permutation importance and SHAP
both include interaction effects, so if interactions were strong you would expect
them to disagree with $S_1$.

In [ ]:
# ==========================================================================
# TODO - do the parameters interact?
#
# Compare the first-order index S1 (a parameter acting alone) with the
# total-order index ST (the same parameter including every interaction it
# takes part in). The difference ST - S1 is the interaction part.
#
# Print the table and the two column sums.
#
# Then answer the question this raises: if the sum of S1 over all nine
# parameters is close to 1, what does that tell you about whether permutation
# importance and SHAP should agree with S1?
#
# Hints:
#   * both columns are already in `merged`
# ==========================================================================


# your code here

---
# Optional extensions (COULD)

Everything above covers the MUST and SHOULD requirements. What follows is
optional. Pick whatever interests your group; you are not expected to do all of
it.

## 13. COULD — other kinds of surrogate

Tree ensembles are not the only option. A **Gaussian process** fits a smooth
function and is often the best choice when data is scarce; a small **neural
network** is the most familiar model from an AI course. Both need their inputs
standardised, because the nine parameters have very different magnitudes
(compare $\beta^* \approx 0.09$ with $\alpha_{k2} \approx 1.0$).

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 14. COULD — does the conclusion survive a change of model settings?

If the recovered ranking flips when you change the number of trees, the ranking
is not a property of the physics — it is an artefact of your model choice. This
is a cheap and very convincing robustness check to show when you present.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 15. COULD — one model for many outputs at once

So far we modelled one scalar. A single multi-output forest can predict all six
at once. Is it as good as six separate models, and is it faster?

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 16. COULD — what would this have saved in CFD time?

Section 11 gave you a number of simulations. Turn it into the number an
engineering manager actually asks for: how much *compute* does the
surrogate-based route save?

You will need a figure for the cost of one CFD simulation — ask your supervisor
for the wall-clock time and core count of a single run in this study. Then
compare the full 1408-simulation analysis against your minimum sample size plus
the (negligible) cost of training and evaluating a surrogate.

State the assumptions on the slide. A saving quoted without them is not a result.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 17. COULD — is the parameter ranking the same at other Reynolds numbers?

The dataset contains the same study at four Reynolds numbers. A conclusion that
only holds at one of them is of limited use to an engineer, so this is a strong
robustness result to report.

Here we give the AI workflow a fixed, small budget at each Reynolds number and
ask whether it reproduces the *trend* found by the full CFD study.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 18. COULD — a completely different geometry

The annular-flow case is the same turbulence model in a curved geometry (flow in
the gap between a rod and an outer cylinder). Loading it is a one-line change,
which is the point: if your pipeline is written cleanly, a new flow case costs
you nothing.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

---
# 19. What you actually hand in

This course has **no written report**. You are assessed on three things:

| when | deliverable |
|---|---|
| before the Christmas break | a **project plan**, agreed with the supervisor |
| week 9 | the **AI solution** — code, results, figures, saved models |
| week 10 | a **final presentation** to your supervisor and the assessors |

So the work below is not material for a report; it is material for a talk and a
clean repository. Plan accordingly: a figure that needs three paragraphs of
explanation is a bad slide.

### For the project plan

Decide early which SHOULD items your group will commit to, and say which ones
you will not do. You have five or six people — split the work by section, not by
"everyone tries the same model". Natural division for Group A: one pair on the
learning curve, one pair on surrogate comparison, one pair on the
surrogate-based Sobol route, everyone on interpretation.

### For the week-9 solution

* a repository that runs end to end from a clean clone;
* your notebooks with outputs, so results can be seen without re-running;
* the figures you will actually present, saved as files;
* a short README stating what you did, what you found, and what you did not do.

### For the final presentation

A suggested spine, roughly one slide each:

1. **The question.** Identifying influential turbulence parameters costs 1408
   CFD simulations. Could far fewer do?
2. **The data and the trap.** The Saltelli design, and why a random split
   flatters you. Show the two R² numbers from section 4a — it is a result, not a
   technicality, and it shows you understood the data.
3. **The surrogate works.** Parity plot, with RMSE / nRMSE / R² and your split
   stated on the slide.
4. **The explanation matches the benchmark.** Your ranking next to the Sobol
   ranking, with error bars.
5. **The learning curve.** Your central result. Explain why accuracy and top-2
   overlap behave differently.
6. **The minimum budget.** How few simulations recover the four-parameter
   subset, and what that saves (sections 11 and 16).
7. **Limits.** See below.
8. **Conclusion** in one sentence that answers slide 1.

### Be honest about the limits — it earns marks, it does not lose them

Specific things that are true of this dataset:

* the surrogate systematically under-estimates the small Sobol indices;
* the ranking among the near-zero parameters is noise, and no amount of data
  fixes it;
* finding the top two parameters is almost trivially easy — say so;
* a high R² does not by itself guarantee a correct explanation;
* unless you did section 16, your conclusion was tested at one Reynolds number.